# EDA Financeira Forense — `Valorajuizado` cruzado com Benner + DeepLegal Processos

Este notebook consolida duas análises em uma única trilha:

1. **EDA principal focada em `Valorajuizado`**.
2. **Extensões analíticas para gerar insights acionáveis para o jurídico**.

## Contexto

A análise parte de um dataframe chamado `df_joined`, já criado no notebook de preparação, contendo o join entre:

- Base Benner de processos jurídicos;
- Base DeepLegal de processos.

O filtro aplicado anteriormente foi:

```python
df_joined["Datainicial"] = pd.to_datetime(df_joined["Datainicial"])

df_joined = df_joined[
    (df_joined["Datainicial"] >= "2015-01-01") &
    (df_joined["Tipoprocesso"] == "PASSIVO") &
    (df_joined["Carteira"].isin(["MASSIFICADO", "MASSIFICADO - REVISIONAIS"]))
]
```

Esse filtro mantém apenas:

- processos iniciados a partir de 2015;
- processos em que o banco é réu/passivo;
- carteiras massificadas e revisionais.

## Objetivo da análise

Responder:

> Dentro da carteira passiva massificada, quais características dos processos explicam maior exposição financeira via `Valorajuizado`?

E, principalmente:

> Quais causas, produtos, fases, territórios, escritórios e tipos de processo fazem mais sentido atacar para reduzir perdas financeiras do banco?

## Observação importante

Este notebook não treina modelo. Ele é uma EDA forense e gerencial, focada em:

- qualidade do valor;
- concentração financeira;
- exposição financeira proxy;
- oportunidades de acordo;
- causas de massa vs causas estratégicas;
- quick wins;
- curvas de ganho financeiro;
- priorização de temas para o jurídico.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

REPORTS_DIR = Path("../outputs/reports")
PROCESSED_DIR = Path("../data/processed")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

VALUE_COL = "Valorajuizado"

print("Setup concluído.")

## 2. Validar entrada

Este notebook assume que `df_joined` já existe em memória.

Se você preferir carregar de um arquivo, descomente e ajuste uma das opções abaixo:

```python
df_joined = pd.read_parquet("../data/processed/df_joined_benner_deeplegal.parquet")
df_joined = pd.read_csv("../data/processed/df_joined_benner_deeplegal.csv")
```

In [ ]:
# Opcional: carregar de arquivo caso o df_joined não esteja em memória
# df_joined = pd.read_parquet("../data/processed/df_joined_benner_deeplegal.parquet")

try:
    df_joined
    print("df_joined encontrado em memória.")
    print("Shape:", df_joined.shape)
except NameError:
    raise NameError(
        "O dataframe df_joined não existe em memória. "
        "Rode antes o notebook de preparação ou carregue o arquivo parquet/csv."
    )

df_joined.head()

## 3. Aplicar ou validar filtro de escopo

Se o filtro já foi aplicado antes, esta célula apenas reforça o recorte.

Recorte:

- `Datainicial >= 2015-01-01`
- `Tipoprocesso == PASSIVO`
- `Carteira in ['MASSIFICADO', 'MASSIFICADO - REVISIONAIS']`

Esse recorte é importante porque evita misturar processos ativos, estratégicos ou carteiras com dinâmica jurídica diferente.

In [ ]:
df_base = df_joined.copy()

if "Datainicial" in df_base.columns:
    df_base["Datainicial"] = pd.to_datetime(df_base["Datainicial"], errors="coerce")

required_filter_cols = ["Datainicial", "Tipoprocesso", "Carteira"]

if all(c in df_base.columns for c in required_filter_cols):
    df_base = df_base[
        (df_base["Datainicial"] >= "2015-01-01") &
        (df_base["Tipoprocesso"] == "PASSIVO") &
        (df_base["Carteira"].isin(["MASSIFICADO", "MASSIFICADO - REVISIONAIS"]))
    ].copy()
else:
    print("Nem todas as colunas do filtro foram encontradas. Mantendo df_joined sem refiltrar.")
    print("Colunas ausentes:", [c for c in required_filter_cols if c not in df_base.columns])

print("Shape após filtro:", df_base.shape)
df_base.head()

## 4. Funções utilitárias

In [ ]:
NULL_STRINGS = {
    "", "null", "nan", "none", "na", "n/a", "-",
    "não informado", "nao informado", "sem informação", "sem informacao"
}

def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    if x in NULL_STRINGS:
        return None
    return x

def save_report(df, filename):
    path = REPORTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"Salvo: {path}")

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def parse_sim_nao(x):
    x = normalize_text(x)
    if x in ["sim", "s", "yes", "true", "1"]:
        return 1
    if x in ["nao", "não", "n", "no", "false", "0"]:
        return 0
    return np.nan

def map_estimativa_perda(x):
    x = normalize_text(x)
    if x is None:
        return np.nan
    if "remoto" in x:
        return 0
    if "possivel" in x or "possível" in x:
        return 1
    if "provavel" in x or "provável" in x:
        return 2
    return np.nan

def peso_estimativa_perda(x):
    x = normalize_text(x)
    if x is None:
        return 0.30
    if "remoto" in x:
        return 0.10
    if "possivel" in x or "possível" in x:
        return 0.50
    if "provavel" in x or "provável" in x:
        return 0.90
    return 0.30

def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def sanitize_filename_col(col):
    return re.sub(r"[^a-zA-Z0-9_]", "_", str(col))

## 5. Preparação da coluna `Valorajuizado`

In [ ]:
df_valor = df_base.copy()

if VALUE_COL not in df_valor.columns:
    raise KeyError(f"A coluna {VALUE_COL} não foi encontrada no dataframe.")

df_valor[VALUE_COL] = pd.to_numeric(df_valor[VALUE_COL], errors="coerce")
df_valor["valor_ajuizado"] = df_valor[VALUE_COL]

df_valor["flag_valor_nulo"] = df_valor["valor_ajuizado"].isna().astype(int)

df_valor["flag_valor_zero_ou_negativo"] = (
    df_valor["valor_ajuizado"].fillna(0) <= 0
).astype(int)

df_valor["valor_ajuizado_log1p"] = np.where(
    df_valor["valor_ajuizado"].fillna(0) > 0,
    np.log1p(df_valor["valor_ajuizado"]),
    0
)

print(df_valor.shape)
df_valor[[VALUE_COL, "valor_ajuizado", "valor_ajuizado_log1p"]].head()

## 6. Diagnóstico inicial do `Valorajuizado`

In [ ]:
valor_summary = df_valor["valor_ajuizado"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

valor_summary.to_frame("valor_ajuizado").to_csv(REPORTS_DIR / "valorajuizado_summary.csv")
valor_summary

## 7. Qualidade do valor

In [ ]:
qualidade_valor = pd.DataFrame([{
    "qtd_processos": len(df_valor),
    "qtd_valor_nao_nulo": df_valor["valor_ajuizado"].notna().sum(),
    "perc_valor_nao_nulo": df_valor["valor_ajuizado"].notna().mean(),
    "qtd_valor_nulo": df_valor["valor_ajuizado"].isna().sum(),
    "perc_valor_nulo": df_valor["valor_ajuizado"].isna().mean(),
    "qtd_valor_zero": (df_valor["valor_ajuizado"] == 0).sum(),
    "perc_valor_zero": (df_valor["valor_ajuizado"] == 0).mean(),
    "qtd_valor_negativo": (df_valor["valor_ajuizado"] < 0).sum(),
    "perc_valor_negativo": (df_valor["valor_ajuizado"] < 0).mean(),
    "valor_total": df_valor["valor_ajuizado"].sum(),
    "valor_medio": df_valor["valor_ajuizado"].mean(),
    "valor_mediano": df_valor["valor_ajuizado"].median(),
    "valor_p95": df_valor["valor_ajuizado"].quantile(0.95),
    "valor_p99": df_valor["valor_ajuizado"].quantile(0.99),
    "valor_maximo": df_valor["valor_ajuizado"].max(),
}])

save_report(qualidade_valor, "qualidade_valorajuizado.csv")
qualidade_valor

## 8. Pareto financeiro geral

In [ ]:
df_valid_value = df_valor[df_valor["valor_ajuizado"].notna() & (df_valor["valor_ajuizado"] > 0)].copy()
df_valid_value = df_valid_value.sort_values("valor_ajuizado", ascending=False)
df_valid_value["rank_valor"] = np.arange(1, len(df_valid_value) + 1)
df_valid_value["rank_percentual"] = df_valid_value["rank_valor"] / len(df_valid_value)
df_valid_value["valor_acumulado"] = df_valid_value["valor_ajuizado"].cumsum()
df_valid_value["perc_valor_acumulado"] = df_valid_value["valor_acumulado"] / df_valid_value["valor_ajuizado"].sum()

pareto_rows = []
for pct in [0.01, 0.05, 0.10, 0.20, 0.30]:
    mask = df_valid_value["rank_percentual"] <= pct
    pareto_rows.append({
        "top_percentual_processos": pct,
        "qtd_processos": int(mask.sum()),
        "valor_total_concentrado": df_valid_value.loc[mask, "valor_ajuizado"].sum(),
        "perc_valor_total_concentrado": df_valid_value.loc[mask, "valor_ajuizado"].sum() / df_valid_value["valor_ajuizado"].sum()
    })

pareto_valor = pd.DataFrame(pareto_rows)
save_report(pareto_valor, "pareto_valorajuizado.csv")
pareto_valor

## 9. Faixas de valor ajuizado

In [ ]:
bins = [-np.inf, 0, 1_000, 5_000, 10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000, np.inf]
labels = [
    "00_sem_valor_ou_zero", "01_ate_1k", "02_1k_5k", "03_5k_10k", "04_10k_25k",
    "05_25k_50k", "06_50k_100k", "07_100k_250k", "08_250k_500k", "09_500k_1mm", "10_acima_1mm"
]

df_valor["faixa_valor_ajuizado"] = pd.cut(
    df_valor["valor_ajuizado"].fillna(0), bins=bins, labels=labels, include_lowest=True
).astype(str)

faixa_valor_report = df_valor.groupby("faixa_valor_ajuizado", dropna=False).agg(
    qtd_processos=("valor_ajuizado", "size"),
    valor_total=("valor_ajuizado", "sum"),
    valor_medio=("valor_ajuizado", "mean"),
    valor_mediano=("valor_ajuizado", "median"),
    valor_p90=("valor_ajuizado", lambda x: x.quantile(0.90)),
    valor_p95=("valor_ajuizado", lambda x: x.quantile(0.95))
).reset_index()

faixa_valor_report["perc_processos"] = faixa_valor_report["qtd_processos"] / faixa_valor_report["qtd_processos"].sum()
faixa_valor_report["perc_valor_total"] = faixa_valor_report["valor_total"] / faixa_valor_report["valor_total"].sum()
save_report(faixa_valor_report, "faixa_valorajuizado_report.csv")
faixa_valor_report

## 10. Valor cruzado com `Estimativa_De_Perda`

Esta é uma das análises centrais. Ela mostra se o valor ajuizado está concentrado em processos classificados como Remoto, Possível ou Provável.

Possíveis insights:

- Valores muito altos em `Remoto` podem indicar necessidade de revisão de provisão.
- Grande concentração em `Possível` ou `Provável` indica foco prioritário para o jurídico.

In [ ]:
if "Estimativa_De_Perda" in df_valor.columns:
    df_valor["estimativa_perda_score"] = df_valor["Estimativa_De_Perda"].apply(map_estimativa_perda)
    df_valor["flag_perda_possivel_ou_provavel"] = (df_valor["estimativa_perda_score"] >= 1).astype(float)
    df_valor.loc[df_valor["estimativa_perda_score"].isna(), "flag_perda_possivel_ou_provavel"] = np.nan
    df_valor["flag_perda_provavel"] = (df_valor["estimativa_perda_score"] == 2).astype(float)
    df_valor.loc[df_valor["estimativa_perda_score"].isna(), "flag_perda_provavel"] = np.nan

    valor_por_estimativa = df_valor.groupby("Estimativa_De_Perda", dropna=False).agg(
        qtd_processos=("valor_ajuizado", "size"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        valor_mediano=("valor_ajuizado", "median"),
        valor_p90=("valor_ajuizado", lambda x: x.quantile(0.90)),
        valor_p95=("valor_ajuizado", lambda x: x.quantile(0.95)),
        valor_p99=("valor_ajuizado", lambda x: x.quantile(0.99))
    ).reset_index()

    valor_por_estimativa["perc_processos"] = valor_por_estimativa["qtd_processos"] / valor_por_estimativa["qtd_processos"].sum()
    valor_por_estimativa["perc_valor_total"] = valor_por_estimativa["valor_total"] / valor_por_estimativa["valor_total"].sum()
    save_report(valor_por_estimativa, "valorajuizado_por_estimativa_perda.csv")
    display(valor_por_estimativa)
else:
    print("Coluna Estimativa_De_Perda não encontrada.")

## 11. Criar exposição financeira proxy

Apenas valor não é suficiente. Um processo de alto valor e remoto pode ter menor prioridade do que um processo médio provável.

Criamos:

```text
exposicao_financeira_proxy = valor_ajuizado × peso_estimativa_perda
```

Pesos exploratórios:

- Remoto: 0.10
- Possível: 0.50
- Provável: 0.90
- Nulo/outros: 0.30

Essa métrica não substitui provisão oficial. Ela é uma régua exploratória para priorização.

In [ ]:
if "Estimativa_De_Perda" in df_valor.columns:
    df_valor["peso_estimativa_perda"] = df_valor["Estimativa_De_Perda"].apply(peso_estimativa_perda)
else:
    df_valor["peso_estimativa_perda"] = 0.30

df_valor["exposicao_financeira_proxy"] = df_valor["valor_ajuizado"].fillna(0) * df_valor["peso_estimativa_perda"]

exposicao_summary = pd.DataFrame([{
    "valor_total_ajuizado": df_valor["valor_ajuizado"].sum(),
    "exposicao_financeira_proxy_total": df_valor["exposicao_financeira_proxy"].sum(),
    "exposicao_media_por_processo": df_valor["exposicao_financeira_proxy"].mean(),
    "exposicao_mediana_por_processo": df_valor["exposicao_financeira_proxy"].median(),
    "exposicao_p90": df_valor["exposicao_financeira_proxy"].quantile(0.90),
    "exposicao_p95": df_valor["exposicao_financeira_proxy"].quantile(0.95),
    "exposicao_p99": df_valor["exposicao_financeira_proxy"].quantile(0.99),
}])

save_report(exposicao_summary, "exposicao_financeira_proxy_summary.csv")
exposicao_summary

## 12. Preparar flag de acordo

In [ ]:
if "Passiveldeacordo" in df_valor.columns:
    df_valor["flag_passivel_acordo"] = df_valor["Passiveldeacordo"].apply(parse_sim_nao)
else:
    print("Coluna Passiveldeacordo não encontrada.")

## 13. Função principal de análise por variável

Esta função cruza `Valorajuizado` e `exposicao_financeira_proxy` com qualquer variável categórica.

In [ ]:
def valor_report_by_group(df, group_col, value_col="valor_ajuizado", exposure_col="exposicao_financeira_proxy", min_count=30):
    if group_col not in df.columns:
        print(f"Coluna não encontrada: {group_col}")
        return None

    agg_dict = {
        "qtd_processos": (value_col, "size"),
        "valor_total": (value_col, "sum"),
        "valor_medio": (value_col, "mean"),
        "valor_mediano": (value_col, "median"),
        "valor_p75": (value_col, lambda x: x.quantile(0.75)),
        "valor_p90": (value_col, lambda x: x.quantile(0.90)),
        "valor_p95": (value_col, lambda x: x.quantile(0.95)),
        "exposicao_total_proxy": (exposure_col, "sum"),
        "exposicao_media_proxy": (exposure_col, "mean"),
    }

    if "flag_perda_possivel_ou_provavel" in df.columns:
        agg_dict["taxa_perda_possivel_ou_provavel"] = ("flag_perda_possivel_ou_provavel", "mean")
    if "flag_perda_provavel" in df.columns:
        agg_dict["taxa_perda_provavel"] = ("flag_perda_provavel", "mean")
    if "flag_passivel_acordo" in df.columns:
        agg_dict["taxa_passivel_acordo"] = ("flag_passivel_acordo", "mean")

    out = df.groupby(group_col, dropna=False).agg(**agg_dict).reset_index()
    out = out[out["qtd_processos"] >= min_count].copy()
    out["perc_processos"] = out["qtd_processos"] / out["qtd_processos"].sum()
    out["share_valor_total"] = out["valor_total"] / out["valor_total"].sum()
    out["share_exposicao_proxy"] = out["exposicao_total_proxy"] / out["exposicao_total_proxy"].sum()
    out["valor_por_processo"] = out["valor_total"] / out["qtd_processos"]
    out = out.sort_values("exposicao_total_proxy", ascending=False)
    return out

## 14. Variáveis relevantes para cruzar com `Valorajuizado`

A lista abaixo combina variáveis Benner e DeepLegal que costumam gerar bons insights para o jurídico.

Como você excluiu várias colunas na preparação, o notebook detecta automaticamente quais ainda existem.

In [ ]:
candidate_group_cols = [
    # Benner
    "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Tipoprocesso", "Uf", "Comarca",
    "Fasedoprocesso", "Situacao", "Estimativa_De_Perda", "Passiveldeacordo", "Processorelevante",
    "Credenciado", "Adv_Interno", "Advogadoadverso", "Nm_Empresa",

    # DeepLegal processos
    "classe_texto", "area_texto", "assunto_normalizado_texto", "assunto_texto", "fase_processual_texto",
    "status_texto", "status_tipo", "cidade", "uf", "vara_texto", "orgao_julgador_texto",
    "juiz_normalizado_texto", "tipo_de_processo_texto", "tipo_de_justica_texto",
    "tipo_de_requerente_texto", "tipo_de_requerido_texto", "resultado_final_processo_text",
    "resultadojulgamento_tipo", "sentenca_tipo", "resultado_do_recurso_texto", "resultado_exec_texto",
]

available_group_cols = [c for c in candidate_group_cols if c in df_valor.columns]
available_group_cols

## 15. Gerar relatórios por variável

In [ ]:
valor_group_reports = {}

for col in available_group_cols:
    print("\n" + "=" * 120)
    print(col)

    report = valor_report_by_group(df_valor, group_col=col, min_count=30)

    if report is not None:
        valor_group_reports[col] = report
        safe_col = sanitize_filename_col(col)
        save_report(report, f"valorajuizado_by_{safe_col}.csv")
        display(report.head(20))

## 16. Ranking dos assuntos prioritários

In [ ]:
if "Nm_Assunto" in valor_group_reports:
    assunto_priority = valor_group_reports["Nm_Assunto"].copy()
    volume_median = assunto_priority["qtd_processos"].median()
    exposicao_median = assunto_priority["exposicao_total_proxy"].median()
    taxa_median = assunto_priority["taxa_perda_possivel_ou_provavel"].median() if "taxa_perda_possivel_ou_provavel" in assunto_priority.columns else np.nan

    assunto_priority["volume_alto"] = assunto_priority["qtd_processos"] >= volume_median
    assunto_priority["exposicao_alta"] = assunto_priority["exposicao_total_proxy"] >= exposicao_median
    assunto_priority["taxa_perda_alta"] = assunto_priority["taxa_perda_possivel_ou_provavel"] >= taxa_median if "taxa_perda_possivel_ou_provavel" in assunto_priority.columns else False

    def classify_priority(row):
        if row["exposicao_alta"] and row["volume_alto"] and row["taxa_perda_alta"]:
            return "prioridade_1_atacar_agora"
        if row["exposicao_alta"] and row["volume_alto"]:
            return "prioridade_2_alta_exposicao_em_massa"
        if row["exposicao_alta"] and not row["volume_alto"]:
            return "prioridade_3_casos_alto_valor"
        if row["volume_alto"] and row["taxa_perda_alta"]:
            return "prioridade_4_revisar_tese_ou_operacao"
        return "monitorar"

    assunto_priority["prioridade"] = assunto_priority.apply(classify_priority, axis=1)
    assunto_priority = assunto_priority.sort_values("exposicao_total_proxy", ascending=False)
    save_report(assunto_priority, "prioridade_financeira_by_nm_assunto.csv")
    display(assunto_priority.head(50))
else:
    print("Nm_Assunto não encontrado nos relatórios.")

## 17. Ranking por ação + assunto

In [ ]:
if "Nm_Acao" in df_valor.columns and "Nm_Assunto" in df_valor.columns:
    df_valor["acao_assunto"] = df_valor["Nm_Acao"].astype(str).fillna("NAO_INFORMADO") + " | " + df_valor["Nm_Assunto"].astype(str).fillna("NAO_INFORMADO")
    acao_assunto_report = valor_report_by_group(df_valor, group_col="acao_assunto", min_count=20)
    save_report(acao_assunto_report, "valorajuizado_by_acao_assunto.csv")
    display(acao_assunto_report.head(50))

## 18. Ranking por produto + assunto

In [ ]:
if "Nm_Produto" in df_valor.columns and "Nm_Assunto" in df_valor.columns:
    df_valor["produto_assunto"] = df_valor["Nm_Produto"].astype(str).fillna("NAO_INFORMADO") + " | " + df_valor["Nm_Assunto"].astype(str).fillna("NAO_INFORMADO")
    produto_assunto_report = valor_report_by_group(df_valor, group_col="produto_assunto", min_count=20)
    save_report(produto_assunto_report, "valorajuizado_by_produto_assunto.csv")
    display(produto_assunto_report.head(50))

## 19. Produto × Ação × Assunto

In [ ]:
if all(c in df_valor.columns for c in ["Nm_Produto", "Nm_Acao", "Nm_Assunto"]):
    df_valor["produto_acao_assunto"] = df_valor["Nm_Produto"].astype(str).fillna("NAO_INFORMADO") + " | " + df_valor["Nm_Acao"].astype(str).fillna("NAO_INFORMADO") + " | " + df_valor["Nm_Assunto"].astype(str).fillna("NAO_INFORMADO")
    produto_acao_assunto_report = valor_report_by_group(df_valor, group_col="produto_acao_assunto", min_count=20)
    save_report(produto_acao_assunto_report, "valorajuizado_by_produto_acao_assunto.csv")
    display(produto_acao_assunto_report.head(50))

## 20. Fase processual

In [ ]:
fase_cols = [c for c in ["Fasedoprocesso", "fase_processual_texto", "status_texto"] if c in df_valor.columns]
for col in fase_cols:
    report = valor_report_by_group(df_valor, group_col=col, min_count=30)
    save_report(report, f"valorajuizado_by_{sanitize_filename_col(col)}.csv")
    print("\n", col)
    display(report.head(30))

## 21. Território: UF, comarca, cidade, vara e órgão julgador

In [ ]:
territory_cols = [c for c in ["Uf", "Comarca", "uf", "cidade", "vara_texto", "orgao_julgador_texto"] if c in df_valor.columns]
for col in territory_cols:
    report = valor_report_by_group(df_valor, group_col=col, min_count=30)
    save_report(report, f"valorajuizado_by_{sanitize_filename_col(col)}.csv")
    print("\n", col)
    display(report.head(30))

## 22. Acordo: valor e oportunidade

In [ ]:
if "Passiveldeacordo" in df_valor.columns:
    acordo_report = valor_report_by_group(df_valor, group_col="Passiveldeacordo", min_count=1)
    save_report(acordo_report, "valorajuizado_by_passiveldeacordo.csv")
    display(acordo_report)

if "flag_passivel_acordo" in df_valor.columns:
    df_valor["flag_oportunidade_acordo_alto_valor"] = ((df_valor["flag_passivel_acordo"] == 1) & (df_valor["valor_ajuizado"] >= df_valor["valor_ajuizado"].quantile(0.75))).astype(int)
    oportunidade_acordo = pd.DataFrame([{
        "qtd_oportunidade_acordo_alto_valor": df_valor["flag_oportunidade_acordo_alto_valor"].sum(),
        "valor_total_oportunidade_acordo": df_valor.loc[df_valor["flag_oportunidade_acordo_alto_valor"] == 1, "valor_ajuizado"].sum(),
        "exposicao_total_oportunidade_acordo": df_valor.loc[df_valor["flag_oportunidade_acordo_alto_valor"] == 1, "exposicao_financeira_proxy"].sum(),
    }])
    save_report(oportunidade_acordo, "oportunidade_acordo_alto_valor.csv")
    display(oportunidade_acordo)

## 23. Segmentação financeira dos processos

In [ ]:
p75_valor = df_valor["valor_ajuizado"].quantile(0.75)
p90_valor = df_valor["valor_ajuizado"].quantile(0.90)
p95_valor = df_valor["valor_ajuizado"].quantile(0.95)

df_valor["segmento_financeiro"] = "massa_baixo_ou_medio_valor"
df_valor.loc[df_valor["valor_ajuizado"] >= p75_valor, "segmento_financeiro"] = "alto_valor_p75"
df_valor.loc[df_valor["valor_ajuizado"] >= p90_valor, "segmento_financeiro"] = "estrategico_top_10pct"
df_valor.loc[df_valor["valor_ajuizado"] >= p95_valor, "segmento_financeiro"] = "critico_top_5pct"

segmento_report = valor_report_by_group(df_valor, group_col="segmento_financeiro", min_count=1)
save_report(segmento_report, "valorajuizado_by_segmento_financeiro.csv")
segmento_report

# Extensões analíticas

## 24. Pareto por grupo

In [ ]:
def pareto_by_group(df, group_col, value_col="valor_ajuizado", top_pct=0.05, min_count=30):
    if group_col not in df.columns:
        return None
    rows = []
    for group, temp in df.groupby(group_col, dropna=False):
        temp = temp[temp[value_col].notna() & (temp[value_col] > 0)].copy()
        if len(temp) < min_count:
            continue
        temp = temp.sort_values(value_col, ascending=False)
        n_top = max(1, int(len(temp) * top_pct))
        total_value = temp[value_col].sum()
        top_value = temp.head(n_top)[value_col].sum()
        rows.append({
            group_col: group,
            "top_pct": top_pct,
            "qtd_processos": len(temp),
            "qtd_top": n_top,
            "valor_total": total_value,
            "valor_top": top_value,
            "perc_valor_top": top_value / total_value if total_value > 0 else np.nan
        })
    return pd.DataFrame(rows).sort_values("perc_valor_top", ascending=False)

pareto_group_cols = existing_cols(df_valor, ["Nm_Produto", "Nm_Assunto", "Nm_Acao", "Fasedoprocesso", "Credenciado", "Comarca", "Uf", "classe_texto", "assunto_normalizado_texto"])
for col in pareto_group_cols:
    report = pareto_by_group(df_valor, col, top_pct=0.05, min_count=30)
    if report is not None and len(report) > 0:
        save_report(report, f"pareto_top5pct_by_{sanitize_filename_col(col)}.csv")
        print("\n", col)
        display(report.head(20))

## 25. Matriz volume × valor médio

In [ ]:
def add_volume_value_quadrant(report):
    out = report.copy()
    qtd_median = out["qtd_processos"].median()
    valor_medio_median = out["valor_medio"].median()
    out["quadrante_volume_valor"] = np.select(
        [
            (out["qtd_processos"] >= qtd_median) & (out["valor_medio"] >= valor_medio_median),
            (out["qtd_processos"] >= qtd_median) & (out["valor_medio"] < valor_medio_median),
            (out["qtd_processos"] < qtd_median) & (out["valor_medio"] >= valor_medio_median),
        ],
        ["alto_volume_alto_valor_medio", "alto_volume_baixo_valor_medio", "baixo_volume_alto_valor_medio"],
        default="baixo_volume_baixo_valor_medio"
    )
    return out

for col in ["Nm_Assunto", "Nm_Produto", "Nm_Acao", "produto_acao_assunto", "acao_assunto"]:
    if col in df_valor.columns:
        base_report = valor_report_by_group(df_valor, col, min_count=20)
        if base_report is not None and len(base_report) > 0:
            quad = add_volume_value_quadrant(base_report)
            save_report(quad, f"quadrante_volume_valor_by_{sanitize_filename_col(col)}.csv")
            print("\n", col)
            display(quad.head(30))

## 26. Análise temporal do valor

In [ ]:
if "Datainicial" in df_valor.columns:
    df_valor["ano_inicio"] = pd.to_datetime(df_valor["Datainicial"], errors="coerce").dt.year
    valor_por_ano = df_valor.groupby("ano_inicio", dropna=False).agg(
        qtd_processos=("valor_ajuizado", "size"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        valor_mediano=("valor_ajuizado", "median"),
        valor_p90=("valor_ajuizado", lambda x: x.quantile(0.90)),
        valor_p95=("valor_ajuizado", lambda x: x.quantile(0.95)),
        exposicao_total_proxy=("exposicao_financeira_proxy", "sum")
    ).reset_index().sort_values("ano_inicio")
    save_report(valor_por_ano, "valorajuizado_por_ano_inicio.csv")
    display(valor_por_ano)

## 27. Aging financeiro

In [ ]:
if "Datainicial" in df_valor.columns:
    data_ref = pd.Timestamp.today().normalize()
    df_valor["idade_processo_dias"] = (data_ref - pd.to_datetime(df_valor["Datainicial"], errors="coerce")).dt.days
    df_valor["faixa_idade_processo"] = pd.cut(
        df_valor["idade_processo_dias"],
        bins=[-np.inf, 180, 365, 730, 1095, 1825, np.inf],
        labels=["ate_6_meses", "6m_1ano", "1_2_anos", "2_3_anos", "3_5_anos", "acima_5_anos"]
    ).astype(str)
    aging_report = valor_report_by_group(df_valor, group_col="faixa_idade_processo", min_count=1)
    save_report(aging_report, "aging_financeiro_by_faixa_idade_processo.csv")
    display(aging_report)

## 28. Valor parado / processos sem movimentação

In [ ]:
if "Qtddias_Ultimoandamento" in df_valor.columns:
    df_valor["Qtddias_Ultimoandamento"] = pd.to_numeric(df_valor["Qtddias_Ultimoandamento"], errors="coerce")
    df_valor["flag_parado_90d"] = (df_valor["Qtddias_Ultimoandamento"] >= 90).astype(int)
    df_valor["flag_parado_180d"] = (df_valor["Qtddias_Ultimoandamento"] >= 180).astype(int)
    df_valor["flag_parado_360d"] = (df_valor["Qtddias_Ultimoandamento"] >= 360).astype(int)
    valor_parado = pd.DataFrame([{
        "valor_total": df_valor["valor_ajuizado"].sum(),
        "valor_parado_90d": df_valor.loc[df_valor["flag_parado_90d"] == 1, "valor_ajuizado"].sum(),
        "valor_parado_180d": df_valor.loc[df_valor["flag_parado_180d"] == 1, "valor_ajuizado"].sum(),
        "valor_parado_360d": df_valor.loc[df_valor["flag_parado_360d"] == 1, "valor_ajuizado"].sum(),
    }])
    save_report(valor_parado, "valor_parado_sem_movimentacao.csv")
    display(valor_parado)
else:
    print("Qtddias_Ultimoandamento não encontrada.")

## 29. Inconsistências financeiras e cadastrais

In [ ]:
p95 = df_valor["valor_ajuizado"].quantile(0.95)
df_valor["flag_alto_valor_remoto"] = ((df_valor["valor_ajuizado"] >= p95) & (df_valor.get("estimativa_perda_score", pd.Series(index=df_valor.index, dtype=float)) == 0)).astype(int)

if "Processorelevante" in df_valor.columns:
    proc_rel_norm = df_valor["Processorelevante"].apply(normalize_text)
    df_valor["flag_alto_valor_nao_relevante"] = ((df_valor["valor_ajuizado"] >= p95) & (~proc_rel_norm.isin(["sim", "s", "yes", "true", "1"]))).astype(int)
else:
    df_valor["flag_alto_valor_nao_relevante"] = np.nan

if "Estimativa_De_Perda" in df_valor.columns:
    df_valor["flag_alto_valor_sem_estimativa"] = ((df_valor["valor_ajuizado"] >= p95) & (df_valor["Estimativa_De_Perda"].isna())).astype(int)
else:
    df_valor["flag_alto_valor_sem_estimativa"] = np.nan

inconsistencias = df_valor[(df_valor["flag_alto_valor_remoto"] == 1) | (df_valor["flag_alto_valor_nao_relevante"] == 1) | (df_valor["flag_alto_valor_sem_estimativa"] == 1)].sort_values("valor_ajuizado", ascending=False)
cols_incons = existing_cols(df_valor, ["Numerodistribuicao", "Identificador", "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Fasedoprocesso", "Estimativa_De_Perda", "Processorelevante", "Passiveldeacordo", "valor_ajuizado", "flag_alto_valor_remoto", "flag_alto_valor_nao_relevante", "flag_alto_valor_sem_estimativa"])
inconsistencias_report = inconsistencias[cols_incons].copy()
save_report(inconsistencias_report, "inconsistencias_financeiras_cadastrais.csv")
inconsistencias_report.head(100)

## 30. Score de atacabilidade

In [ ]:
df_valor["score_atacabilidade"] = 0
df_valor["score_atacabilidade"] += np.where(df_valor["valor_ajuizado"] >= df_valor["valor_ajuizado"].quantile(0.75), 1, 0)

if "flag_passivel_acordo" in df_valor.columns:
    df_valor["score_atacabilidade"] += df_valor["flag_passivel_acordo"].fillna(0)

if "Fasedoprocesso" in df_valor.columns:
    fase_norm = df_valor["Fasedoprocesso"].astype(str).str.lower()
    df_valor["score_atacabilidade"] += np.where(fase_norm.str.contains("conhecimento|inicial|julgamento", na=False), 1, 0)
    df_valor["score_atacabilidade"] -= np.where(fase_norm.str.contains("execu|cumprimento|arquivado", na=False), 1, 0)

df_valor["score_atacabilidade"] = df_valor["score_atacabilidade"].clip(lower=0)
impacto_alto = df_valor["exposicao_financeira_proxy"] >= df_valor["exposicao_financeira_proxy"].median()
atacavel_alto = df_valor["score_atacabilidade"] >= df_valor["score_atacabilidade"].median()

df_valor["matriz_impacto_atacabilidade"] = np.select(
    [impacto_alto & atacavel_alto, impacto_alto & ~atacavel_alto, ~impacto_alto & atacavel_alto],
    ["alto_impacto_alta_atacabilidade", "alto_impacto_baixa_atacabilidade", "baixo_impacto_alta_atacabilidade"],
    default="baixo_impacto_baixa_atacabilidade"
)

atacabilidade_report = valor_report_by_group(df_valor, group_col="matriz_impacto_atacabilidade", min_count=1)
save_report(atacabilidade_report, "matriz_impacto_atacabilidade.csv")
atacabilidade_report

## 31. Cenários de atuação

In [ ]:
def scenario_summary(df, mask, scenario_name):
    temp = df[mask].copy()
    return {
        "cenario": scenario_name,
        "qtd_processos": len(temp),
        "perc_processos": len(temp) / len(df),
        "valor_total": temp["valor_ajuizado"].sum(),
        "perc_valor_total": temp["valor_ajuizado"].sum() / df["valor_ajuizado"].sum(),
        "exposicao_total_proxy": temp["exposicao_financeira_proxy"].sum(),
        "perc_exposicao_total_proxy": temp["exposicao_financeira_proxy"].sum() / df["exposicao_financeira_proxy"].sum()
    }

scenarios = []
scenarios.append(scenario_summary(df_valor, df_valor["valor_ajuizado"] >= df_valor["valor_ajuizado"].quantile(0.95), "top_5pct_valor"))
scenarios.append(scenario_summary(df_valor, df_valor["exposicao_financeira_proxy"] >= df_valor["exposicao_financeira_proxy"].quantile(0.95), "top_5pct_exposicao_proxy"))

if "flag_passivel_acordo" in df_valor.columns:
    scenarios.append(scenario_summary(df_valor, (df_valor["flag_passivel_acordo"] == 1) & (df_valor["valor_ajuizado"] >= df_valor["valor_ajuizado"].quantile(0.75)), "passivel_acordo_acima_p75"))

if "assunto_priority" in globals() and "Nm_Assunto" in df_valor.columns:
    assuntos_p1 = assunto_priority.loc[assunto_priority["prioridade"] == "prioridade_1_atacar_agora", "Nm_Assunto"].tolist()
    scenarios.append(scenario_summary(df_valor, df_valor["Nm_Assunto"].isin(assuntos_p1), "assuntos_prioridade_1"))

scenario_report = pd.DataFrame(scenarios)
save_report(scenario_report, "cenario_atuacao_financeira.csv")
scenario_report

## 32. Quick wins

In [ ]:
df_valor["flag_quick_win"] = 0

conditions = (
    (df_valor["valor_ajuizado"] >= df_valor["valor_ajuizado"].quantile(0.75)) &
    (df_valor.get("flag_passivel_acordo", pd.Series(0, index=df_valor.index)).fillna(0) == 1) &
    (df_valor.get("flag_perda_possivel_ou_provavel", pd.Series(0, index=df_valor.index)).fillna(0) == 1)
)

if "Fasedoprocesso" in df_valor.columns:
    fase = df_valor["Fasedoprocesso"].astype(str).str.lower()
    conditions = conditions & ~fase.str.contains("execu|arquivado|cumprimento", na=False)

df_valor.loc[conditions, "flag_quick_win"] = 1
quick_win_cols = existing_cols(df_valor, ["Numerodistribuicao", "Identificador", "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Fasedoprocesso", "Estimativa_De_Perda", "Passiveldeacordo", "Credenciado", "Adv_Interno", "valor_ajuizado", "exposicao_financeira_proxy", "score_atacabilidade", "matriz_impacto_atacabilidade"])
quick_wins = df_valor[df_valor["flag_quick_win"] == 1].sort_values("exposicao_financeira_proxy", ascending=False)[quick_win_cols]
save_report(quick_wins, "quick_wins_acordo_alto_valor.csv")
quick_wins.head(100)

## 33. Divergência Benner × DeepLegal

In [ ]:
if "valor_valor" in df_valor.columns:
    df_valor["valor_deeplegal"] = pd.to_numeric(df_valor["valor_valor"], errors="coerce")
    df_valor["abs_diff_valor_benner_dl"] = (df_valor["valor_ajuizado"] - df_valor["valor_deeplegal"]).abs()
    df_valor["diff_relativa_benner_dl"] = safe_divide(df_valor["abs_diff_valor_benner_dl"], df_valor[["valor_ajuizado", "valor_deeplegal"]].max(axis=1))
    df_valor["flag_divergencia_alta_benner_dl"] = ((df_valor["abs_diff_valor_benner_dl"] >= 10_000) & (df_valor["diff_relativa_benner_dl"] >= 0.50)).astype(int)

    divergencia_summary = pd.DataFrame([{
        "qtd_com_ambos_valores": int((df_valor["valor_ajuizado"].notna() & df_valor["valor_deeplegal"].notna()).sum()),
        "correlacao_benner_deeplegal": df_valor[["valor_ajuizado", "valor_deeplegal"]].corr().iloc[0, 1],
        "qtd_divergencia_alta": int(df_valor["flag_divergencia_alta_benner_dl"].sum()),
        "perc_divergencia_alta": float(df_valor["flag_divergencia_alta_benner_dl"].mean()),
        "media_abs_diff": float(df_valor["abs_diff_valor_benner_dl"].mean()),
        "mediana_abs_diff": float(df_valor["abs_diff_valor_benner_dl"].median()),
        "p95_abs_diff": float(df_valor["abs_diff_valor_benner_dl"].quantile(0.95)),
    }])
    save_report(divergencia_summary, "divergencia_valor_benner_deeplegal_summary.csv")
    display(divergencia_summary)

    for col in existing_cols(df_valor, ["Nm_Assunto", "Nm_Produto", "Nm_Acao", "Carteira", "Comarca", "Fasedoprocesso", "Credenciado"]):
        divergencia_by_group = df_valor.groupby(col, dropna=False).agg(
            qtd_processos=("valor_ajuizado", "size"),
            media_abs_diff=("abs_diff_valor_benner_dl", "mean"),
            mediana_abs_diff=("abs_diff_valor_benner_dl", "median"),
            taxa_divergencia_alta=("flag_divergencia_alta_benner_dl", "mean")
        ).reset_index().query("qtd_processos >= 30").sort_values("media_abs_diff", ascending=False)
        save_report(divergencia_by_group, f"divergencia_valor_benner_dl_by_{sanitize_filename_col(col)}.csv")
        print("\nDivergência por", col)
        display(divergencia_by_group.head(20))
else:
    print("Coluna valor_valor da DeepLegal não encontrada.")

## 34. Deep dive automático dos top assuntos

In [ ]:
if "assunto_priority" in globals() and "Nm_Assunto" in df_valor.columns:
    top_assuntos = assunto_priority.head(10)["Nm_Assunto"].tolist()
    deep_dive_rows = []
    for assunto in top_assuntos:
        temp = df_valor[df_valor["Nm_Assunto"] == assunto]
        def mode_or_none(col):
            if col not in temp.columns:
                return None
            m = temp[col].mode(dropna=True)
            return m.iloc[0] if not m.empty else None
        deep_dive_rows.append({
            "Nm_Assunto": assunto,
            "qtd_processos": len(temp),
            "valor_total": temp["valor_ajuizado"].sum(),
            "exposicao_total_proxy": temp["exposicao_financeira_proxy"].sum(),
            "valor_medio": temp["valor_ajuizado"].mean(),
            "produto_top": mode_or_none("Nm_Produto"),
            "acao_top": mode_or_none("Nm_Acao"),
            "fase_top": mode_or_none("Fasedoprocesso"),
            "comarca_top": mode_or_none("Comarca"),
            "credenciado_top": mode_or_none("Credenciado"),
            "estimativa_perda_top": mode_or_none("Estimativa_De_Perda"),
            "passivel_acordo_top": mode_or_none("Passiveldeacordo"),
        })
    deep_dive_top_assuntos = pd.DataFrame(deep_dive_rows)
    save_report(deep_dive_top_assuntos, "deep_dive_top_10_assuntos.csv")
    display(deep_dive_top_assuntos)
else:
    print("assunto_priority ou Nm_Assunto não disponível.")

## 35. Curva de ganho financeiro

In [ ]:
df_gain = df_valor.sort_values("exposicao_financeira_proxy", ascending=False).copy()
df_gain["rank"] = np.arange(1, len(df_gain) + 1)
df_gain["perc_processos"] = df_gain["rank"] / len(df_gain)
df_gain["valor_acumulado"] = df_gain["valor_ajuizado"].cumsum()
df_gain["perc_valor_acumulado"] = df_gain["valor_acumulado"] / df_gain["valor_ajuizado"].sum()
df_gain["exposicao_acumulada"] = df_gain["exposicao_financeira_proxy"].cumsum()
df_gain["perc_exposicao_acumulada"] = df_gain["exposicao_acumulada"] / df_gain["exposicao_financeira_proxy"].sum()

gain_summary = []
for pct in [0.01, 0.05, 0.10, 0.20, 0.30]:
    temp = df_gain[df_gain["perc_processos"] <= pct]
    gain_summary.append({
        "top_percentual_processos": pct,
        "qtd_processos": len(temp),
        "valor_capturado": temp["valor_ajuizado"].sum(),
        "perc_valor_capturado": temp["valor_ajuizado"].sum() / df_valor["valor_ajuizado"].sum(),
        "exposicao_capturada": temp["exposicao_financeira_proxy"].sum(),
        "perc_exposicao_capturada": temp["exposicao_financeira_proxy"].sum() / df_valor["exposicao_financeira_proxy"].sum(),
    })

gain_summary = pd.DataFrame(gain_summary)
save_report(gain_summary, "curva_ganho_financeiro_summary.csv")
save_report(df_gain, "curva_ganho_financeiro_detalhada.csv")
gain_summary

## 36. Ranking final de processos por exposição

In [ ]:
ranking_cols = [c for c in [
    "Numerodistribuicao", "Identificador", "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto",
    "Tipoprocesso", "Uf", "Comarca", "Fasedoprocesso", "Situacao", "Estimativa_De_Perda",
    "Passiveldeacordo", "Processorelevante", "Credenciado", "Adv_Interno", "valor_ajuizado",
    "faixa_valor_ajuizado", "segmento_financeiro", "peso_estimativa_perda", "exposicao_financeira_proxy",
    "score_atacabilidade", "matriz_impacto_atacabilidade", "flag_quick_win", "classe_texto",
    "assunto_normalizado_texto", "fase_processual_texto", "status_texto", "cidade", "uf", "vara_texto", "orgao_julgador_texto"
] if c in df_valor.columns]

ranking_processos_valor = df_valor[ranking_cols].sort_values(["exposicao_financeira_proxy", "valor_ajuizado"], ascending=False)
save_report(ranking_processos_valor, "ranking_processos_por_valorajuizado_exposicao.csv")
ranking_processos_valor.head(100)

## 37. Resumo executivo da EDA

In [ ]:
executive_summary_valor = pd.DataFrame([{
    "qtd_processos_analisados": len(df_valor),
    "valor_total_ajuizado": df_valor["valor_ajuizado"].sum(),
    "valor_medio_ajuizado": df_valor["valor_ajuizado"].mean(),
    "valor_mediano_ajuizado": df_valor["valor_ajuizado"].median(),
    "valor_p90_ajuizado": df_valor["valor_ajuizado"].quantile(0.90),
    "valor_p95_ajuizado": df_valor["valor_ajuizado"].quantile(0.95),
    "valor_p99_ajuizado": df_valor["valor_ajuizado"].quantile(0.99),
    "exposicao_financeira_proxy_total": df_valor["exposicao_financeira_proxy"].sum(),
    "qtd_processos_top_5pct_valor": (df_valor["segmento_financeiro"] == "critico_top_5pct").sum(),
    "valor_total_top_5pct": df_valor.loc[df_valor["segmento_financeiro"] == "critico_top_5pct", "valor_ajuizado"].sum(),
    "qtd_oportunidade_acordo_alto_valor": df_valor["flag_oportunidade_acordo_alto_valor"].sum() if "flag_oportunidade_acordo_alto_valor" in df_valor.columns else np.nan,
    "qtd_quick_wins": df_valor["flag_quick_win"].sum() if "flag_quick_win" in df_valor.columns else np.nan
}])
save_report(executive_summary_valor, "executive_summary_eda_valorajuizado.csv")
executive_summary_valor

## 38. Salvar base analítica final

In [ ]:
df_valor.to_parquet(PROCESSED_DIR / "abt_eda_valorajuizado_benner_deeplegal_processos.parquet", index=False)
ranking_processos_valor.to_parquet(PROCESSED_DIR / "ranking_processos_por_valorajuizado_exposicao.parquet", index=False)
if "assunto_priority" in globals():
    assunto_priority.to_parquet(PROCESSED_DIR / "prioridade_financeira_by_nm_assunto.parquet", index=False)
print("Base salva em:", PROCESSED_DIR / "abt_eda_valorajuizado_benner_deeplegal_processos.parquet")
print("Ranking salvo em:", PROCESSED_DIR / "ranking_processos_por_valorajuizado_exposicao.parquet")

# Roteiro de explicação para o jurídico

## 1. Escopo analisado

A análise foi feita sobre processos:

- iniciados a partir de 2015;
- em que o banco é réu/passivo;
- das carteiras massificadas e revisionais.

## 2. Pergunta respondida

A pergunta central foi:

> Quais causas/processos concentram maior exposição financeira e devem ser priorizados pelo jurídico?

## 3. Principais blocos da análise

### Concentração financeira

Identifica se poucos processos concentram grande parte do valor ajuizado.

### Assuntos prioritários

Mostra quais assuntos combinam alto volume, alto valor, alta exposição proxy e alta taxa de perda possível/provável.

### Produto × ação × assunto

Ajuda a encontrar causa raiz operacional, conectando produto do banco, tipo de demanda e objeto jurídico.

### Fase processual

Permite diferenciar estratégia:

- fase inicial: tese e documentação;
- recurso: estratégia recursal;
- execução: contenção financeira/acordo;
- arquivado: análise retrospectiva.

### Território

Mostra concentração por UF, comarca, cidade, vara e órgão julgador.

### Acordo

Identifica oportunidades de acordo em processos de alto valor.

### Aging e valor parado

Mostra quanto valor está em processos antigos ou sem movimentação.

### Quick wins

Lista processos com alto valor, passíveis de acordo e perda possível/provável.

### Curva de ganho financeiro

Mostra quanto valor/exposição é capturado se o jurídico atacar apenas os top X% processos.

## 4. Como usar os resultados

A recomendação é usar três entregáveis principais:

1. `prioridade_financeira_by_nm_assunto.csv`  
   Para decidir quais temas atacar.

2. `valorajuizado_by_produto_acao_assunto.csv`  
   Para entender causa raiz e desenhar plano de ação.

3. `ranking_processos_por_valorajuizado_exposicao.csv`  
   Para priorização operacional de processos individuais.

## 5. Próximo passo

Depois de validar esta EDA com o jurídico, o próximo passo é criar uma ABT de feature engineering para modelagem, separando claramente:

- score atual de carteira;
- modelo preditivo sem leakage;
- features disponíveis no momento de previsão;
- variáveis de resultado que não podem entrar como preditoras.